## 01 - Audio Embeddings 
In this notebook, we will load audio files, convert them to embedding vectors (using mean pooling), and export to json files

In [1]:
# !pip install torch

In [2]:
# !pip install transformers

In [3]:
# !pip install flash-attn

In [4]:
# !pip install librosa

In [5]:
import pandas as pd
import torch
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
import librosa
import os
import numpy as np

In [6]:
# testing torch
print(torch.cuda.is_available())
print(torch.cuda.current_device())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

True
0
cuda


In [7]:
paths = ["../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/ad/", 
         "../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/cn/", 
         "../../data/Test Dataset/Diagnosis Testing dataset/diagnosis/test-dist/audio/",
        ]

In [8]:
def get_file_list(path, ext=".wav"):
    """
        get all filenames in a folder
        returns a dictionary {file_id:full_path}
    """
    result = dict()
    # filenames = []
    # get only wav files
    print("Reading from folder =", path)
    files = [item
             for item in os.listdir(path)
             if item.endswith(ext)
            ]
    
    for fname in files:
        result[fname.replace(ext, "")] = path + fname 

    print("Count =", len(result))
    return result


In [9]:
file_paths = [get_file_list(path)
              for path in paths
             ]
print(len(file_paths))
print(len(file_paths[0].keys()))


Reading from folder = ../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/ad/
Count = 87
Reading from folder = ../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/cn/
Count = 79
Reading from folder = ../../data/Test Dataset/Diagnosis Testing dataset/diagnosis/test-dist/audio/
Count = 71
3
87


In [10]:
# merge dictionaries
file_paths_dict = dict()


for fpath in file_paths:
    file_paths_dict.update(fpath)

print(len(file_paths_dict))

237


In [11]:
# create audio processing model
model_name = "facebook/wav2vec2-large-960h-lv60-self"
model = Wav2Vec2Model.from_pretrained(model_name,
                                      torch_dtype=torch.float16,
                                     ).to(device)


C:\Users\Minh UBC\miniconda3\Lib\site-packages\torch\_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1', 'wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

In [13]:
# put into a function
def audio2vec(fname, model=model, extractor=extractor, fl16=True):
    """
        Read a wav file and return mean pooling embeddings (length 512)
    """
    result = []
    print("Loading file =", fname)
    input_audio, sample_rate = librosa.load(fname, sr=16000)

    # convert to float16
    if fl16:
        input_audio = input_audio.astype(np.float16)
        
    print("Completed")
    print("Calculating embeddings...")
    
    result = extractor(input_audio, return_tensors="pt", sampling_rate=sample_rate).to(device)
    with torch.no_grad():
        output = model(result.input_values)

    print("Calculating mean pooling...")

    # result = np.average(output.extract_features[0], axis=0)
    result = torch.mean(output.extract_features[0], dim=0).cpu()
    # debug:
    print("Shape =", result.shape)
    return result

In [14]:
# test loading wav file
# fname = "../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/cn/adrso002.wav"
# path = "../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/"
# fname = "../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/cn/adrso002.wav"  # small file 6 MB
# fname = "../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/ad/adrso024.wav" # bigger file 20 MB
# fname = "../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/ad/adrso192.wav" # bigger file 38 MB

VEC_LEN = 512

fname = file_paths_dict["adrso002"] # small file 
print(fname)

result = audio2vec(fname)
assert len(result) == VEC_LEN
print("All passed")

../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/cn/adrso002.wav
Loading file = ../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/cn/adrso002.wav
Completed
Calculating embeddings...
Calculating mean pooling...
Shape = torch.Size([512])
All passed


In [15]:
# load previous embed, will skip already processed items
path_out = "../../data/audio_embed/"
fname_existing = "train_audio_embed.json"

try:
    existing_df = pd.read_json(path_out + fname_existing,
                              orient="records")
except Exception as e:
    print("Error reading file =", fname_existing)
    existing_df = pd.DataFrame({"file_id":[], 
                                "audio_embed":[]
                               })
print(len(existing_df))
existing_df.head()

237


,file_id,audio_embed
0,adrsdt26,"[0.572265625, 0.0052719116, 0.1276855469, 0.04..."
1,adrsdt32,"[0.4912109375, -0.3137207031, 0.4453125, -0.19..."
2,adrsdt37,"[0.5161132812, -0.2147216797, 0.3342285156, 0...."
3,adrsdt44,"[0.3947753906, 0.0309295654, 1.111328125, 0.22..."
4,adrsdt46,"[0.3186035156, -0.2010498047, -0.1440429688, -..."


In [16]:
# check existing file
fid = "adrso024"
print(sum(existing_df["file_id"].str.contains(fid)))
print(fid in list(existing_df["file_id"]))
print(existing_df.query("file_id == @fid").iloc[0]["audio_embed"][:10]) # check existing value

1
True
[0.0742797852, -0.2470703125, -0.2939453125, -0.1875, 0.5258789062, 0.6860351562, -0.1779785156, -0.0234375, -0.2900390625, -0.1143188477]


In [17]:
# calculate embeddings for list of files
embed_data = dict()
n = len(file_paths_dict.items())
print("N =", n)

max_n = n # set to n for full set

fnames = list(file_paths_dict.keys())
for i, fname in enumerate(fnames[:max_n]):
    print(f"Processing {i+1}/{max_n} = {(i+1)*100/max_n:.0f} %, file name = {fname}")
    if fname in list(existing_df["file_id"]):
        print("File existed. Skipping...")
        # embed_data[fname] = existing_df.query("file_id == @fid")["audio_embed"][0] # copy over from existing dataframe
        continue
    
    v = audio2vec(file_paths_dict[fname])
    embed_data[fname] = v
    print("---------------")

print("All Completed")

N = 237
Processing 1/237 = 0 %, file name = adrso024
File existed. Skipping...
Processing 2/237 = 1 %, file name = adrso025
File existed. Skipping...
Processing 3/237 = 1 %, file name = adrso027
File existed. Skipping...
Processing 4/237 = 2 %, file name = adrso028
File existed. Skipping...
Processing 5/237 = 2 %, file name = adrso031
File existed. Skipping...
Processing 6/237 = 3 %, file name = adrso032
File existed. Skipping...
Processing 7/237 = 3 %, file name = adrso033
File existed. Skipping...
Processing 8/237 = 3 %, file name = adrso035
File existed. Skipping...
Processing 9/237 = 4 %, file name = adrso036
File existed. Skipping...
Processing 10/237 = 4 %, file name = adrso039
File existed. Skipping...
Processing 11/237 = 5 %, file name = adrso043
File existed. Skipping...
Processing 12/237 = 5 %, file name = adrso045
File existed. Skipping...
Processing 13/237 = 5 %, file name = adrso046
File existed. Skipping...
Processing 14/237 = 6 %, file name = adrso047
File existed. Skipp

In [18]:
# print("Count =", len(embed_data))
# print("Element shape =", embed_data[fnames[0]].shape)

In [19]:
f_get_embed = lambda fid: (np.asarray(embed_data[fid].cpu().numpy()) 
                           # if fid in embed_data.keys()
                           # else existing_df.query("file_id==@fid")["audio_embed"][0]
                          )

In [20]:
# convert to dataframe

df_out = pd.DataFrame({"file_id": list(embed_data.keys())})
# df_out["audio_embed"] = df_out["file_id"].apply(lambda fid: np.asarray(embed_data.[fid].cpu().numpy()))
# df_out["audio_embed"] = df_out["file_id"].apply(lambda fid: list(torch.asarray(embed_data[fid])))
df_out["audio_embed"] = df_out["file_id"].apply(f_get_embed)

df_out.head()

,file_id,audio_embed


In [21]:
print(len(df_out))

0


In [22]:
# copy existing data over
df_out = pd.concat([df_out, existing_df], ignore_index=True)
print(len(df_out))
    

237


In [23]:
# check type
print(type(df_out.iloc[0]["audio_embed"]))
print(type(df_out.iloc[0]["audio_embed"][0]))

<class 'list'>
<class 'float'>


In [24]:
# write to json file
path_out = "../../data/audio_embed/"
fname_out = "train_audio_embed.json"

print("Writing to file =", fname_out)
df_out.to_json(path_out + fname_out,
               orient="records"
              )
print("Completed")

Writing to file = train_audio_embed.json
Completed
